# Notebook 45 — Quantization and Deployment Model Formats

    ## Learning objectives

    - Distinguish weight, activation, and KV-cache quantization across training and serving
- Compare bitsandbytes, GPTQ, AWQ, FP8, GGUF, and Safetensors without conflating format and method
- Design calibration and quality-performance evaluations before producing deployment artifacts

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = ['transformers>=4.51,<5', 'accelerate>=1.6', 'safetensors>=0.5']

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
if IN_COLAB:
    from google.colab import userdata
    token = None
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            pass
        if token:
            break
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGINGFACE_TOKEN"] = token
else:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Hugging Face token configured:", bool(os.getenv("HUGGINGFACE_TOKEN")))


## 45.1 Quantization changes representation and sometimes computation

Quantization represents values with fewer bits or restricted numeric formats to reduce memory bandwidth,
capacity, and sometimes latency. Weight-only quantization compresses parameters while activations remain at
higher precision. Weight-activation schemes quantize both. KV-cache quantization targets memory that grows with
active tokens during serving. Optimizer-state quantization is primarily a training concern.

“4-bit” is incomplete: method, group size, symmetric/asymmetric scales, zero points, compute dtype, outlier
handling, kernel, hardware, and model architecture determine quality and speed. Packed weights reduce storage
even when a backend dequantizes them during compute; real speedups require compatible kernels and shapes.


In [ ]:
def weight_memory(params_b, bits): return params_b * 1e9 * bits / 8 / 2**30
for size in [0.5, 7, 70]:
    print(f"{size:g}B", {f"{bits}-bit GiB": round(weight_memory(size, bits), 2) for bits in [16, 8, 4]})
print("Scales, zero-points, metadata, temporary buffers, and KV cache are additional.")


## 45.2 Numeric model

Uniform affine quantization maps values approximately as `q = clamp(round(x/scale) + zero_point)` and
reconstructs `x_hat = scale*(q-zero_point)`. Per-channel scales preserve different output-channel ranges;
groupwise scales trade metadata and kernel complexity for lower error. Symmetric quantization simplifies zero
points. Non-uniform codebooks such as NF4 allocate representable values according to assumed distributions.

Post-training quantization (PTQ) transforms an existing checkpoint. Quantization-aware training simulates
quantization during optimization. Dynamic methods calculate some scales at runtime; static methods calibrate
them. Outlier-aware approaches retain sensitive values or channels at higher precision.


In [ ]:
import torch
torch.manual_seed(0)
x = torch.randn(256) * 1.7
qmin, qmax = -7, 7
scale = x.abs().max() / qmax
q = torch.clamp(torch.round(x / scale), qmin, qmax)
restored = q * scale
print({"scale": scale.item(), "unique_codes": q.unique().numel(),
       "mae": (x-restored).abs().mean().item(), "max_error": (x-restored).abs().max().item()})


## 45.3 Method and container are different layers

Safetensors is a safe, efficiently loadable tensor container; it does not imply a precision. Transformers
repositories combine weights with config, tokenizer, chat template, and generation metadata. bitsandbytes
provides runtime 8/4-bit loading and training workflows such as QLoRA. GPTQ approximates weights using
calibration data and second-order information. AWQ protects salient weights based on activation observations.
FP8 uses floating formats supported efficiently on newer accelerators.

GGUF is a deployment container associated with llama.cpp-family local runtimes and can carry multiple
quantization types plus metadata. A file extension cannot guarantee the correct prompt template, tokenizer,
license, or runtime compatibility. Preserve the canonical source revision and conversion command with every
derivative artifact.


In [ ]:
matrix = [
    ("Safetensors", "tensor container", "HF/serving ecosystems", "not a quantizer"),
    ("bitsandbytes", "runtime/training library", "Transformers/QLoRA", "backend-dependent kernels"),
    ("GPTQ/AWQ", "PTQ methods + formats", "GPU inference", "needs calibration/compatible engine"),
    ("FP8", "numeric formats", "modern accelerators", "hardware and scale strategy matter"),
    ("GGUF", "deployment container", "llama.cpp/Ollama-style local serving", "conversion metadata matters"),
]
for row in matrix: print(" | ".join(row))


## 45.4 Calibration and sensitivity

Calibration samples should represent deployment languages, domains, lengths, modalities, chat templates, and
activation outliers. Hundreds of copied generic sentences may optimize the wrong distribution. Keep calibration
data separate from quality evaluation and record its provenance. Layer sensitivity varies: embeddings, output
heads, attention projections, and outlier-heavy layers may need higher precision.

Compare the quantized artifact with the exact unquantized parent on fixed logits/perplexity and downstream
behavior. Evaluate rare tokens, long context, structured output, tools, reasoning, safety, multilingual content,
and calibration. A small average benchmark delta can conceal a severe critical-slice regression.


In [ ]:
def regression(candidate, baseline, higher_is_better=True):
    delta = candidate - baseline
    return delta if higher_is_better else -delta
results = {"task_accuracy": regression(.812, .821),
           "schema_validity": regression(.991, .997),
           "p95_latency_improvement": regression(1.4, 2.1, higher_is_better=False)}
print(results, "quality gate:", results["task_accuracy"] >= -.01 and results["schema_validity"] >= -.005)


## 45.5 Hugging Face loading pattern

Transformers integrates quantization configurations, but support depends on model, hardware, Accelerate,
library versions, and the installed backend. The guarded pattern below documents intent without pretending a
CPU or Colab runtime supports every kernel. Device mapping and compute dtype affect both memory and numerics.
Saving runtime-quantized modules may differ from producing a portable pre-quantized repository; consult the
pinned backend documentation and test reload in the target engine.


In [ ]:
RUN_4BIT_LOAD = False
if RUN_4BIT_LOAD:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    model_id = "Qwen/Qwen2.5-0.5B-Instruct"
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                               bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=quant, device_map="auto")
    print(model.get_memory_footprint())
else:
    print("4-bit load skipped; requires a supported accelerator and bitsandbytes installation.")


## 45.6 Benchmark, publish, and roll back

Measure artifact bytes, peak resident memory, maximum sustainable concurrent tokens, TTFT, inter-token latency,
total tokens/second, energy/cost, cold-load time, and quality. Warm up kernels and report hardware, engine,
batch/concurrency, prompt/output distributions, context limit, speculative settings, and quantization metadata.
Smaller weights may allow larger batches, so single-request latency and fleet throughput can disagree.

Publish a derivative model card linking the parent commit, license, calibration set description, conversion
tool/version/command, hashes, supported engines/hardware, evaluation deltas, and known limitations. Scan artifacts
for secrets and untrusted custom code. Canary the new artifact and keep the prior image and weights available.
Quantizing adapters and merging them in the wrong order can change results; record the exact composition graph.


## 45.7 Quantization error is layer- and task-dependent

Weight-only, weight-plus-activation, static, dynamic, post-training, and quantization-aware methods make different assumptions. Group size, calibration data, outlier treatment, scale/zero-point representation, and kernel support matter as much as nominal bit width. Compare logits or activations layerwise to locate error, then run perplexity, task, safety, and structured-output suites. A small average weight error can damage a sensitive capability, while a larger error in another layer may be harmless.


In [ ]:
torch.manual_seed(2); w=torch.randn(64,64); scale=w.abs().max()/127; q=(w/scale).round().clamp(-127,127); restored=q*scale; error=(restored-w); print("RMSE",error.square().mean().sqrt().item(),"max",error.abs().max().item(),"relative",error.norm().item()/w.norm().item())


## 45.8 Format does not guarantee runtime speed

Safetensors, GGUF, GPTQ, AWQ, bitsandbytes, and other names describe different storage or quantization ecosystems; speed comes from compatible kernels, hardware, shapes, and serving behavior. Measure load time, resident memory, peak memory, prefill and decode throughput, TTFT, concurrency, and quality on the target engine. Include KV-cache dtype and context distribution. Preserve the original checkpoint and conversion command, validate metadata and tokenizer/template, and test reload before deleting intermediates.


In [ ]:
measurements=[{"format":"fp16","gib":14,"tok_s":45,"quality":.82},{"format":"int4","gib":5,"tok_s":62,"quality":.79}];
for m in measurements: print(m["format"],"tokens/s/GiB",m["tok_s"]/m["gib"],"quality",m["quality"])


## Primary references and further study

Use the pinned library documentation that matches your environment. Papers explain the method and assumptions; current official documentation defines the executable API.

- [GPTQ](https://arxiv.org/abs/2210.17323)
- [AWQ](https://arxiv.org/abs/2306.00978)
- [Transformers quantization](https://huggingface.co/docs/transformers/quantization/overview)


## Exercises

    1. Quantize a small model with two methods and compare memory, latency, perplexity, and task slices.
2. Build a calibration set that covers long context, code, multilingual text, and chat templates.
3. Write a derivative model card containing every conversion and rollback artifact.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
